# 🏛️ ĐỒ ÁN MÔN HỌC DAT301m - STAGE 4: MAX CAPACITY & COMPLETE CONVERGENCE
## Mở rộng Dữ liệu Sát trần 9.500 Ảnh & Huấn luyện Hội tụ Hoàn toàn 15 Epochs

* **Mục tiêu:** Khai thác tối đa hạn mức dữ liệu (ràng buộc đề bài $\le$ 10.000 ảnh) và tăng thời gian huấn luyện lên 15 epochs để mạng VGG16 scratch đạt trạng thái hội tụ tối ưu.
* **Can thiệp Data:**
  1. Tăng quy mô lên **9.500 ảnh** (950 ảnh/lớp, chừa biên an toàn 500 ảnh).
  2. Áp dụng toàn bộ chuỗi chuẩn hóa: Khử rò rỉ MD5 + Lọc nhiễu nhãn + Căn giữa BBox + Tẩy trắng nền + Augmentation không lật ngang.
  3. Huấn luyện 15 epochs.


In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Đang chạy trên Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Đang chạy trên Local Machine")

if IN_COLAB:
    if os.path.exists("/content/deeplearningai-data-centric-competition"):
        os.chdir("/content/deeplearningai-data-centric-competition")
    elif not os.path.exists("data"):
        os.system("git clone https://github.com/Nov05/deeplearningai-data-centric-competition.git")
        if os.path.exists("deeplearningai-data-centric-competition"):
            os.chdir("deeplearningai-data-centric-competition")

# Giải nén data và label_book nếu chưa có
import tarfile
if not os.path.exists("data") and os.path.exists("original/data.tar.gz"):
    with tarfile.open("original/data.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu data thành công.")

if not os.path.exists("label_book") and os.path.exists("original/label_book.tar.gz"):
    with tarfile.open("original/label_book.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu label_book thành công.")

import glob
raw_count = len(glob.glob("data/**/*.png", recursive=True))
print(f"✅ Dữ liệu gốc: Tìm thấy {raw_count} ảnh.")
assert raw_count > 0, "LỖI: Chưa có ảnh trong thư mục 'data/'!"


🚀 Đang chạy trên Google Colab


/tmp/ipykernel_609/3864034174.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


✅ Đã giải nén dữ liệu data thành công.
✅ Dữ liệu gốc: Tìm thấy 2880 ảnh.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import hashlib
import shutil
from PIL import Image

IMAGE_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASS_NAMES = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
RANDOM_SEED = 123
EPOCHS = 15 # Huấn luyện chuẩn 15 epochs đồng bộ cho mọi Stage

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU ({gpus[0].name}) -> Tốc độ huấn luyện cực nhanh!")
else:
    print("⚠️ Thiết bị hiện tại: CPU (Khuyên dùng: Trên Colab vào Menu Runtime -> Change runtime type -> Chọn T4 GPU)")

def build_fixed_vgg16_model():
    """
    KIẾN TRÚC VGG16 TỰ XÂY DỰNG TỪ ĐẦU (FROM SCRATCH) THEO YÊU CẦU GIẢNG VIÊN:
    - KHÔNG dùng thư viện có sẵn tf.keras.applications.VGG16
    - KHÔNG dùng pretrained weights ImageNet
    - Tự code tay đúng 13 lớp Conv2D + 5 lớp MaxPooling2D + 3 lớp Dense = 16 Layers VGG16
    - Khởi tạo He-Normal, cố định bất biến qua mọi Stage
    """
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name="input_image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescaling")(inputs)

    # Block 1 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c1")(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b1_pool")(x)

    # Block 2 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c1")(x)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b2_pool")(x)

    # Block 3 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c1")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c2")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b3_pool")(x)

    # Block 4 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b4_pool")(x)

    # Block 5 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b5_pool")(x)

    # Classifier Head (13 Conv + 3 Dense = 16 Layers VGG16)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc1")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout1")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc2")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout2")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Fixed_VGG16_Scratch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).")


TensorFlow Version: 2.21.0
🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU (/physical_device:GPU:0) -> Tốc độ huấn luyện cực nhanh!
✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).


In [ ]:
def preprocess_and_center_numeral(img_pil, target_size=(32, 32), pad_ratio=0.15):
    gray = img_pil.convert("L")
    arr = np.array(gray)
    arr[arr > 215] = 255 # Tẩy trắng nền scan
    ys, xs = np.where(arr < 215)
    if len(ys) == 0:
        return None
    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()
    h = ymax - ymin + 1
    w = xmax - xmin + 1
    max_dim = max(h, w)
    pad = int(max_dim * pad_ratio)
    total_size = max_dim + 2 * pad
    canvas = np.ones((total_size, total_size), dtype=np.uint8) * 255
    crop = arr[ymin:ymax+1, xmin:xmax+1]
    canvas[(total_size-h)//2:(total_size-h)//2+h, (total_size-w)//2:(total_size-w)//2+w] = crop
    canvas[canvas < 120] = (canvas[canvas < 120] * 0.7).astype(np.uint8) # Boost tương phản
    res = Image.fromarray(canvas).resize(target_size, Image.Resampling.BILINEAR)
    return res.convert("RGB")

def ensure_clean_label_book():
    CLEAN_LB_DIR = "clean_label_book"
    if not os.path.exists(CLEAN_LB_DIR) or len(glob.glob(f"{CLEAN_LB_DIR}/*/*.png")) == 0:
        os.makedirs(CLEAN_LB_DIR, exist_ok=True)
        for c in CLASS_NAMES:
            os.makedirs(os.path.join(CLEAN_LB_DIR, c), exist_ok=True)
            for f in glob.glob(f"label_book/{c}/*.png"):
                proc = preprocess_and_center_numeral(Image.open(f), target_size=IMAGE_SIZE)
                if proc is not None:
                    proc.save(os.path.join(CLEAN_LB_DIR, c, os.path.basename(f)))
        print("✅ Đã tự động khởi tạo clean_label_book chuẩn hóa.")


## 1. NẠP DỮ LIỆU TỐI ƯU HÓA HOÀN TOÀN (9.500 ẢNH CÂN BẰNG - TỰ ĐỘNG SINH NẾU CHƯA CÓ)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

AUGMENTED_DIR = "enhanced_data_max"
ensure_clean_label_book()

if not os.path.exists(AUGMENTED_DIR) or len(glob.glob(f"{AUGMENTED_DIR}/*/*.png")) < 9000:
    print("Đang tự động sinh tập dữ liệu 9.500 ảnh tối ưu hóa...")
    os.makedirs(AUGMENTED_DIR, exist_ok=True)
    datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.08, height_shift_range=0.08, zoom_range=0.08, fill_mode="nearest")
    for c in CLASS_NAMES:
        cdest = os.path.join(AUGMENTED_DIR, c)
        os.makedirs(cdest, exist_ok=True)
        raws = glob.glob(f"data/train/{c}/*.png") + glob.glob(f"data/val/{c}/*.png")
        sh = set()
        v_imgs = []
        for f in raws:
            if os.path.basename(f) in {"aac26b40-ce5d-11eb-b317-38f9d35ea60f.png", "b0a13faa-ce5d-11eb-b317-38f9d35ea60f.png"}:
                continue
            with open(f, "rb") as fp:
                h = hashlib.md5(fp.read()).hexdigest()
                if h in sh: continue
                sh.add(h)
            p = preprocess_and_center_numeral(Image.open(f), target_size=IMAGE_SIZE)
            if p is not None:
                v_imgs.append(np.array(p))
        for i, arr_img in enumerate(v_imgs):
            Image.fromarray(arr_img).save(os.path.join(cdest, f"clean_{i}.png"))
        needed = 950 - len(v_imgs)
        if needed > 0 and len(v_imgs) > 0:
            it = datagen.flow(np.array(v_imgs), batch_size=1, shuffle=True, seed=RANDOM_SEED)
            for i in range(needed):
                Image.fromarray(next(it)[0].astype(np.uint8)).save(os.path.join(cdest, f"aug_{i}.png"))

train_ds_s4 = tf.keras.preprocessing.image_dataset_from_directory(
    AUGMENTED_DIR, validation_split=0.2, subset="training", seed=RANDOM_SEED,
    labels="inferred", label_mode="categorical", class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

val_ds_s4 = tf.keras.preprocessing.image_dataset_from_directory(
    AUGMENTED_DIR, validation_split=0.2, subset="validation", seed=RANDOM_SEED,
    labels="inferred", label_mode="categorical", class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

clean_lb_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "clean_label_book", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=False,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

print(f"Tổng số ảnh trong tập tối ưu: {len(glob.glob(f'{AUGMENTED_DIR}/*/*.png'))} ảnh.")


✅ Đã tự động khởi tạo clean_label_book chuẩn hóa.
Đang tự động sinh tập dữ liệu 9.500 ảnh tối ưu hóa...
Found 9500 files belonging to 10 classes.
Using 7600 files for training.
Found 9500 files belonging to 10 classes.
Using 1900 files for validation.
Found 52 files belonging to 10 classes.
Tổng số ảnh trong tập tối ưu: 9500 ảnh.


## 2. HUẤN LUYỆN 15 EPOCHS ĐỂ ĐẠT HỘI TỤ ĐỈNH CAO


In [ ]:
EPOCHS_S4 = 15 # Huấn luyện chuẩn 15 epochs đồng bộ cho mọi Stage

model_s4 = build_fixed_vgg16_model()

# Callbacks thông minh: Lưu epoch tốt nhất và tự dừng khi không tăng điểm (Chuẩn DeepLearning.AI)
callbacks_s4 = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model_stage4.weights.h5",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]
history_s4 = model_s4.fit(
    train_ds_s4,
    validation_data=val_ds_s4,
    epochs=EPOCHS_S4,
    verbose=1,
    callbacks=callbacks_s4
)

val_loss_s4, val_acc_s4 = model_s4.evaluate(val_ds_s4, verbose=0)
lb_loss_s4, lb_acc_s4 = model_s4.evaluate(clean_lb_ds, verbose=0)

print("="*60)
print(f"📊 KẾT QUẢ STAGE 4 (MAX CAPACITY 9.500 ẢNH & 15 EPOCHS):")
print(f"  • Validation Accuracy: {val_acc_s4*100:.2f}% | Loss: {val_loss_s4:.4f}")
print(f"  • Label Book Accuracy: {lb_acc_s4*100:.2f}% | Loss: {lb_loss_s4:.4f}")
print("="*60)

model_s4.save_weights("best_vgg16_stage4.weights.h5")
print("✅ Đã lưu trọng số mô hình tốt nhất: best_vgg16_stage4.weights.h5")


Epoch 1/15
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.0993 - loss: 2.3846
Epoch 1: val_accuracy improved from None to 0.21000, saving model to best_model_stage4.weights.h5

Epoch 1: finished saving model to best_model_stage4.weights.h5
238/238 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.1045 - loss: 2.3191 - val_accuracy: 0.2100 - val_loss: 2.2706
Epoch 2/15
237/238 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1977 - loss: 2.1312
Epoch 2: val_accuracy improved from 0.21000 to 0.49211, saving model to best_model_stage4.weights.h5

Epoch 2: finished saving model to best_model_stage4.weights.h5
238/238 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.2616 - loss: 1.9459 - val_accuracy: 0.4921 - val_loss: 1.3673
Epoch 3/15
237/238 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4874 - loss: 1.3815
Epoch 3: val_accuracy improved from 0.49211 to 0.76632, saving model to best_model_stage4.weights.h5

Epoch 3: finished saving model to best_model_stage4.weights.h5
238/238 ━